# Approximation Algorithms for Steiner Trees in Weighted Graphs

Nikola Labus — Naučno izračunavanje 2025/26

In [1]:
import networkx as nx
import time
from itertools import combinations
from pathlib import Path

## STP Parser

In [2]:
def parse_stp(filepath):
    G = nx.Graph()
    terminals = []
    name = ""
    section = None

    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('33D32945'):
                continue

            if line.startswith('SECTION'):
                section = line.split()[1]
                continue
            if line == 'END' or line == 'EOF':
                section = None
                continue

            if section == 'Comment':
                if line.startswith('Name'):
                    name = line.split('"')[1]

            elif section == 'Graph':
                if line.startswith('Nodes'):
                    n = int(line.split()[1])
                    G.add_nodes_from(range(1, n + 1))
                elif line.startswith('E '):
                    parts = line.split()
                    u, v, w = int(parts[1]), int(parts[2]), int(parts[3])
                    G.add_edge(u, v, weight=w)

            elif section == 'Terminals':
                if line.startswith('T '):
                    terminals.append(int(line.split()[1]))

    return G, terminals, name

## Brute Force (Egzaktan algoritam)

Za svaki podskup neterminalnih čvorova proveravamo da li se terminali mogu povezati
kroz indukovani podgraf. Pamtimo minimalno razapinjuće stablo sa najmanjom težinom.

In [4]:
def brute_force_steiner(G, terminals):
    terminal_set = set(terminals)
    non_terminals = [v for v in G.nodes() if v not in terminal_set]
    best_weight = float('inf')
    best_tree = None
    subsets_checked = 0

    for k in range(len(non_terminals) + 1):
        for subset in combinations(non_terminals, k):
            subsets_checked += 1
            nodes = terminal_set | set(subset)
            subgraph = G.subgraph(nodes)

            if nx.is_connected(subgraph):
                mst = nx.minimum_spanning_tree(subgraph)
                weight = mst.size(weight='weight')
                if weight < best_weight:
                    best_weight = weight
                    best_tree = mst

    return best_tree, best_weight, subsets_checked

## Test on Small Examples

In [5]:
small_files = sorted(Path('data/small').glob('*.stp'))

for filepath in small_files:
    G, terminals, name = parse_stp(filepath)
    print(f'--- {name} ---')
    print(f'Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}, Terminals: {len(terminals)}')
    print(f'Non-terminals: {G.number_of_nodes() - len(terminals)}')
    print(f'Subsets to check: 2^{G.number_of_nodes() - len(terminals)} = {2 ** (G.number_of_nodes() - len(terminals))}')

    start = time.time()
    tree, weight, subsets = brute_force_steiner(G, terminals)
    elapsed = time.time() - start

    steiner_points = [v for v in tree.nodes() if v not in terminals]
    print(f'Optimal weight: {weight}')
    print(f'Steiner points used: {steiner_points}')
    print(f'Edges: {list(tree.edges(data=True))}')
    print(f'Subsets checked: {subsets}')
    print(f'Time: {elapsed:.4f}s')
    print()

--- small01 ---
Nodes: 5, Edges: 7, Terminals: 3
Non-terminals: 2
Subsets to check: 2^2 = 4
Optimal weight: 7.0
Steiner points used: [2, 4]
Edges: [(1, 2, {'weight': 2}), (2, 4, {'weight': 1}), (2, 3, {'weight': 2}), (4, 5, {'weight': 2})]
Subsets checked: 4
Time: 0.0017s

--- small02 ---
Nodes: 10, Edges: 15, Terminals: 4
Non-terminals: 6
Subsets to check: 2^6 = 64
Optimal weight: 19.0
Steiner points used: [2, 5, 8, 9]
Edges: [(1, 2, {'weight': 3}), (2, 5, {'weight': 4}), (4, 5, {'weight': 2}), (5, 8, {'weight': 3}), (7, 8, {'weight': 2}), (8, 9, {'weight': 2}), (9, 10, {'weight': 3})]
Subsets checked: 64
Time: 0.0036s

--- small03 ---
Nodes: 15, Edges: 25, Terminals: 5
Non-terminals: 10
Subsets to check: 2^10 = 1024
Optimal weight: 32.0
Steiner points used: [2, 4, 6, 8, 10, 11, 13]
Edges: [(1, 2, {'weight': 3}), (2, 4, {'weight': 4}), (4, 6, {'weight': 2}), (4, 5, {'weight': 3}), (6, 8, {'weight': 3}), (8, 9, {'weight': 2}), (8, 10, {'weight': 3}), (10, 11, {'weight': 2}), (10, 13, {